# 🤖 Working with AI APIs — OpenAI 
### AI Bootcamp · Day 3 · DigiPen Institute of Technology

> **How to use this notebook:**  
> Read each section. Run each code cell with **Shift + Enter**. Do NOT skip ahead.
> Every cell builds on the previous one.

| Topic | What you learn |
|---|---|
| 📡 Topic 1 | What is an API? JSON basics |
| 🧠 Topic 2 | Tokens, messages, temperature |
| 🔑 Topic 3 | Getting your API key safely |
| 🔌 Topic 4 | Your first real AI call |
| 💻 Topic 5 | Safe function + streaming |
| 🚀 Topic 6 | Build a personality bot + memory bot |

**Cost for today:** Less than $0.05 USD for the entire session.  
**Requirement:** OpenAI account + API key (credit card needed — see Topic 3).


---
# 📡 Topic 1 — What is an API?

**API** stands for **A**pplication **P**rogramming **I**nterface.

Think of it like a restaurant:
- **YOU** (your Python code) = the customer
- **MENU** (API documentation) = the list of things you can ask for
- **WAITER** (the API) = takes your request, brings back the result
- **KITCHEN** (OpenAI's server) = does the actual AI work — you never see inside

**Key insight:** ChatGPT is just an API with a nice website on top.  
Today you will call that SAME API directly from Python — no website needed.

### ⚠️ Important: LLMs can HALLUCINATE
LLMs sometimes generate text that sounds confident but is factually wrong.  
Always verify important facts. Treat the AI as a brilliant but occasionally unreliable assistant.


## 1.1 JSON — The Language APIs Speak

Every piece of data going TO an API, and every response coming BACK, is in **JSON** format.

JSON looks almost exactly like a Python dictionary. Key differences:
- Always **double quotes** for keys: `"name"` not `'name'`
- Lowercase: `true`, `false`, `null` — NOT Python's `True`, `False`, `None`
- No trailing comma after the last item

**Two functions you must know:**
- `json.dumps()` = Python dict → JSON string (for **SENDING** to an API)
- `json.loads()` = JSON string → Python dict (for **RECEIVING** from an API)


In [1]:
# Cell 1.1: JSON basics — run this cell with Shift+Enter
import json   # built-in Python library — no pip install needed

# A normal Python dictionary
student = {
    "name":    "Aisha",   # str  — text value
    "year":    2,                        # int  — whole number
    "gpa":     3.85,                     # float — decimal number
    "active":  True,                     # bool — Python True becomes JSON true
    "modules": ["CS2030", "CS2040"],     # list — becomes JSON array
    "advisor": None                      # None — becomes JSON null
}

# ── SEND: convert Python dict → JSON string ──────────────────────────
# This is what LEAVES your code and travels to the API server
json_string = json.dumps(student, indent=2)
# indent=2 = pretty-print with 2-space indentation (easier to read)

print("=== STEP 1: What we SEND to the API ===")
print(json_string)
print(f'Type: {type(json_string)}  ← it is a STRING, not a dict!')
print()

# ── RECEIVE: convert JSON string → Python dict ───────────────────────
# This is what comes BACK from the API
api_response = '{"answer":"Python is great!","tokens":23}'
parsed = json.loads(api_response)
# json.loads() converts the JSON string back into a Python dict

print("=== STEP 2: What we RECEIVE from the API ===")
print(f'Answer: {parsed["answer"]}')
print(f'Tokens: {parsed["tokens"]}')
print(f'Type:   {type(parsed)}  ← now a DICT!')
print()

# ── VERIFY: round-trip check — did all fields survive? ───────────────
received_back = json.loads(json_string)
print("=== STEP 3: Round-Trip Verification ===")
fields = ["name", "year", "gpa", "active", "modules", "advisor"]
for field in fields:
    original = student[field]
    restored = received_back[field]
    status = "OK" if original == restored else "MISMATCH"
    print(f'  {field:<10}: {status}  | {original!r}')
print()
print("All fields survived! JSON is lossless.")


=== STEP 1: What we SEND to the API ===
{
  "name": "Aisha",
  "year": 2,
  "gpa": 3.85,
  "active": true,
  "modules": [
    "CS2030",
    "CS2040"
  ],
  "advisor": null
}
Type: <class 'str'>  ← it is a STRING, not a dict!

=== STEP 2: What we RECEIVE from the API ===
Answer: Python is great!
Tokens: 23
Type:   <class 'dict'>  ← now a DICT!

=== STEP 3: Round-Trip Verification ===
  name      : OK  | 'Aisha'
  year      : OK  | 2
  gpa       : OK  | 3.85
  active    : OK  | True
  modules   : OK  | ['CS2030', 'CS2040']
  advisor   : OK  | None

All fields survived! JSON is lossless.


## 🧪 Self-Check — Topic 1
Answer these in your head before moving on:
1. What does API stand for? Explain each word.
2. What is the difference between `json.dumps()` and `json.loads()`?
3. Fix this broken JSON: `{'name': 'James', 'active': True, 'score': None,}`
4. Name 3 apps you use that rely on APIs.


---
# 🧠 Topic 2 — How LLM APIs Work

Before writing real API code, you need to understand 3 concepts:
**Tokens**, **Messages**, and **Temperature**.


## 2.1 Tokens — The Currency of Everything

LLMs do **NOT** process words. They process **tokens** — sub-word chunks of text.

**Rule of thumb:** 1 token ≈ 0.75 English words, or 1 word ≈ 1.33 tokens

**Why 1.33?**  
Average English word = ~4.5 characters.  
GPT tokeniser encodes ~3.4 characters per token.  
So: tokens per word = 4.5 ÷ 3.4 = **1.32 ≈ 1.33**

**Why it matters:** Cost, speed, and context limits are ALL measured in tokens.


In [3]:
# Cell 2.1: Count tokens using tiktoken — the real OpenAI tokeniser
# !pip install tiktoken   ← run this if you get ModuleNotFoundError
import tiktoken

# Load the tokeniser for GPT-4o — this is the REAL tokeniser OpenAI uses
enc = tiktoken.encoding_for_model('gpt-4o')
print('Tokeniser loaded!')

# Test texts — from short words to full sentences
texts = [
    'hello',
    'running',
    'ChatGPT',              # splits into 3 tokens: Chat + G + PT
    'supercalifragilistic', # splits into 5 tokens
    'DigiPen',
    'What is machine learning?',
    'The bootcamp teaches DigiPen students to build AI apps.',
]

print(f"{'Text':<52} | Tokens | Words | Ratio")
print('-' * 75)

for text in texts:
    tok_ids = enc.encode(text)       # convert text to list of token IDs
    tok     = len(tok_ids)           # count IDs = number of tokens = what you pay for
    words   = len(text.split())      # split on spaces = word count
    ratio   = tok / words if words > 0 else 0
    print(f'{text:<52} | {tok:>6} | {words:>5} | {ratio:.2f}')

print()
print('Rule: token count ≈ word count × 1.33')

# BONUS: see the actual token IDs for ChatGPT
word = 'ChatGPT'
ids  = enc.encode(word)
print(f'Token IDs for "{word}": {ids}')
print(f'That is why it costs {len(ids)} tokens, not 1.')


Tokeniser loaded!
Text                                                 | Tokens | Words | Ratio
---------------------------------------------------------------------------
hello                                                |      1 |     1 | 1.00
running                                              |      1 |     1 | 1.00
ChatGPT                                              |      2 |     1 | 2.00
supercalifragilistic                                 |      6 |     1 | 6.00
DigiPen                                              |      3 |     1 | 3.00
What is machine learning?                            |      5 |     4 | 1.25
The bootcamp teaches DigiPen students to build AI apps. |     12 |     9 | 1.33

Rule: token count ≈ word count × 1.33
Token IDs for "ChatGPT": [14065, 162016]
That is why it costs 2 tokens, not 1.


## 2.2 The Messages Array — How the AI Reads Your Request

The OpenAI API does NOT take a plain text string.  
It takes a **list of message dictionaries**, each with a `role` and `content`.

| Role | Purpose | Example |
|---|---|---|
| `system` | AI's rules and persona — applies to every reply | `'You are a helpful tutor.'` |
| `user` | Your question or instruction | `'What is Python?'` |
| `assistant` | The AI's previous replies — used to simulate memory | `'Python is...'` |

### ⚡ CRITICAL: LLMs are COMPLETELY STATELESS
Every API call starts with **zero memory**.  
The model has no idea what you asked 5 seconds ago.  
**The messages array IS the only memory.**  
Include past turns → AI can remember. Skip them → AI forgets everything.


In [4]:
# Cell 2.2: Demonstrate statelessness — no API key needed
# We use a fake_api_call() to show the concept before using the real API

def fake_api_call(messages):
    '''Simulates an AI. Returns hardcoded replies based on what we send.
    In Topic 4 this becomes: client.chat.completions.create(...)'''
    last = messages[-1]['content']
    if 'Priya' in last:
        return 'Nice to meet you, Priya! CS at DigiPen is a great choice!'
    elif 'name' in last.lower() and len(messages) == 1:
        return 'I do not know your name. You have not told me.'
    elif 'name' in last.lower() and len(messages) > 1:
        return 'Your name is Priya! You told me in your first message.'
    return 'I can help with that.'

# ── TURN 1 ───────────────────────────────────────────────────────────
print('TURN 1 — User introduces themselves')
messages_turn1 = [
    {'role': 'user', 'content': 'Hi! My name is Priya and I study CS at DigiPen.'}
]
reply1 = fake_api_call(messages_turn1)
print(f'You: {messages_turn1[0]["content"]}')
print(f'Bot: {reply1}')

# ── TURN 2 WRONG: no history ─────────────────────────────────────────
print()
print('TURN 2 WRONG — New call, no history')
messages_bad = [{'role': 'user', 'content': 'What is my name?'}]
print(f'Bot: {fake_api_call(messages_bad)}')
print('The AI forgot Priya! Each API call starts with zero memory.')

# ── TURN 2 CORRECT: include full history ─────────────────────────────
print()
print('TURN 2 CORRECT — Full history included')
messages_good = [
    {'role': 'user',      'content': 'Hi! My name is Priya and I study CS at DigiPen.'},
    {'role': 'assistant', 'content': reply1},   # include the AI's Turn 1 reply
    {'role': 'user',      'content': 'What is my name?'}
]
print(f'Bot: {fake_api_call(messages_good)}')
print('The AI remembered! Because we included the history.')


TURN 1 — User introduces themselves
You: Hi! My name is Priya and I study CS at DigiPen.
Bot: Nice to meet you, Priya! CS at DigiPen is a great choice!

TURN 2 WRONG — New call, no history
Bot: I do not know your name. You have not told me.
The AI forgot Priya! Each API call starts with zero memory.

TURN 2 CORRECT — Full history included
Bot: Your name is Priya! You told me in your first message.
The AI remembered! Because we included the history.


## 2.3 Temperature — The Creativity Dial

| Value | Behaviour | Use for |
|---|---|---|
| `0.0` | Same answer every time (deterministic) | Maths, code, facts, JSON output |
| `0.7` | Balanced — **start here for most tasks** | General chat, explanations, tutoring |
| `1.0` | More creative and varied | Brainstorming, storytelling |
| `1.5+` | Very creative, may be inconsistent | Comedy, poetry, experimental |

> **Run the temperature experiment in Topic 4 Cell 4 after you have your API key.**


## 🧪 Self-Check — Topic 2
1. How many tokens is a 500-word essay approximately?
2. What are the 3 message roles? What does each one do?
3. You ask the AI your name and it says it does not know. What went wrong?
4. Should a maths quiz checker use temperature 0.0 or 1.5? Why?


---
# 🔑 Topic 3 — Getting Your API Key

## 3.1 Sign Up for OpenAI
1. Go to **platform.openai.com**
2. Click **Sign Up** — use your DigiPen email
3. Profile icon → **API Keys** → **Create new secret key** → name it `AIBootcamp2026`
4. **COPY IT IMMEDIATELY** — you only see it once. Starts with `sk-proj-...`

## 3.2 Set a Spending Limit — DO THIS FIRST
> ⚠️ A runaway loop can make thousands of calls. Set a hard limit before anything else.

**Settings → Billing → Usage Limits**
- Hard Limit = **$5** (account STOPS at this amount)
- Soft Limit = **$3** (you get a warning email)

## 3.3 Create Your .env File
Create a file called `.env` in the **same folder as this notebook**.  
Open it in Notepad (NOT Jupyter) and add:
```
OPENAI_API_KEY=sk-proj-your-actual-key-here
```
Rules: no spaces around `=`, no quotes, no `.txt` extension.

## 3.4 Create Your .gitignore File
Create a file called `.gitignore` in the same folder. Add:
```
.env
venv/
__pycache__/
```
This tells git to never commit your key. **Never put your key directly in code.**


In [1]:
# Cell 3.1: Verify your setup — run this before Topic 4
from dotenv import load_dotenv
import os

# load_dotenv() reads the .env file and puts the key into the environment
# MUST run before creating the OpenAI client
load_dotenv()

key = os.getenv('OPENAI_API_KEY')
if key:
    # Only show first 8 characters — never print the full key!
    print(f'Key found! First 8 chars: {key[:8]}...')
    print('You are ready for Topic 4!')
else:
    print('Key NOT found. Check:')
    print('  1. Does .env exist in the SAME folder as this notebook?')
    print('  2. Is it named .env (not .env.txt)?')
    print('  3. Is OPENAI_API_KEY spelled correctly? (no spaces, all caps)')
    print('  4. Did you save the .env file after editing it?')
    print()
    print('Run this to see files in your folder:')
    print(os.listdir('.'))


Key found! First 8 chars: sk-proj-...
You are ready for Topic 4!


---
# 🔌 Topic 4 — Your First Real API Call

This is the moment. We call a real AI from Python.
Follow along and type every character — do not copy-paste.


In [2]:
# Cell 4.1: Setup — run this first in EVERY notebook that uses OpenAI

# Import the OpenAI client class
from openai import OpenAI

# Import the function that reads our .env file
from dotenv import load_dotenv

import os

# CRITICAL: load .env BEFORE creating OpenAI()
# This puts OPENAI_API_KEY into the environment
# If you call OpenAI() first, it cannot find the key → AuthenticationError
load_dotenv()

# Create the client — automatically reads OPENAI_API_KEY from environment
# Every API call goes through this object
client = OpenAI()

print('Client ready! We are connected to OpenAI.')


Client ready! We are connected to OpenAI.


In [3]:
# Cell 4.2: Your first real AI call — every parameter explained

response = client.chat.completions.create(

    # Which AI model to use
    # gpt-4o-mini = fast, cheap (~$0.15 per million input tokens)
    model='gpt-4o-mini',

    # The conversation — a list of message dicts
    messages=[
        {
            'role':    'system',
            # system = the AI's rules, applied to EVERY reply
            'content': 'You are a friendly assistant. Answer in exactly 2 sentences.'
        },
        {
            'role':    'user',
            # user = your question
            'content': 'What is Python programming language?'
        }
    ],

    # Creativity dial — 0.7 is balanced, use this as your default
    temperature=0.7,

    # ALWAYS set this — prevents unexpectedly long (expensive) responses
    max_tokens=100
)

# Navigate the response object to get the text:
# .choices = list of responses (usually just 1)
# [0]      = take the first one
# .message = the AI's message object
# .content = the actual text string
answer = response.choices[0].message.content
print('AI says:')
print(answer)
print()

# Always check finish_reason:
# 'stop'   = complete response ✅
# 'length' = CUT OFF by max_tokens — no error raised, you MUST check this!
finish = response.choices[0].finish_reason
print(f'Finish reason: {finish}')

# Token usage and cost
in_t  = response.usage.prompt_tokens
out_t = response.usage.completion_tokens
cost  = in_t * 0.00000015 + out_t * 0.0000006
print(f'Tokens: {in_t} in + {out_t} out')
print(f'Cost:   ${cost:.8f} USD')


AI says:
Python is a high-level, interpreted programming language known for its clear syntax and readability, making it accessible for beginners and experienced developers alike. It supports multiple programming paradigms, including procedural, object-oriented, and functional programming, and has a rich ecosystem of libraries and frameworks for various applications.

Finish reason: stop
Tokens: 30 in + 57 out
Cost:   $0.00003870 USD


## 4.3 Understanding the Response Object

| Path | What it gives you |
|---|---|
| `response.choices[0].message.content` | The AI's reply as a string |
| `response.choices[0].finish_reason` | `'stop'` = complete, `'length'` = cut off |
| `response.usage.prompt_tokens` | Tokens YOU sent (input) |
| `response.usage.completion_tokens` | Tokens AI replied with (output) |

> ⚠️ **`finish_reason = 'length'` is a silent bug.** No error is raised.  
> The response is just cut off mid-sentence. Always check this after every call.


In [5]:
# Cell 4.3: Temperature experiment — same prompt, different temperatures
import time

def temp_test(temperature, prompt, runs=4):
    print(f'temperature = {temperature}')
    print('-' * 45)
    results = []
    for i in range(1, runs + 1):
        response = client.chat.completions.create(
            model='gpt-4o-mini',
            messages=[{'role': 'user', 'content': prompt}],
            temperature=temperature,
            max_tokens=10,
        )
        answer = response.choices[0].message.content.strip()
        results.append(answer)
        print(f'  Run {i}: {answer}')
        time.sleep(1)
    if len(set(results)) == 1:
        print('  → Both runs IDENTICAL (expected at low temperature)')
    else:
        print('  → Runs DIFFERENT (expected at high temperature)')
    print()

prompt = 'Give me exactly ONE word meaning happy. Only the word.'
temp_test(0.0, prompt)   # always the same
temp_test(0.7, prompt)   # usually similar
temp_test(1.5, prompt)   # often different


temperature = 0.0
---------------------------------------------
  Run 1: Joyful
  Run 2: Joyful
  Run 3: Joyful
  Run 4: Joyful
  → Both runs IDENTICAL (expected at low temperature)

temperature = 0.7
---------------------------------------------
  Run 1: Joyful
  Run 2: Joyful
  Run 3: Joyful.
  Run 4: Joyful
  → Runs DIFFERENT (expected at high temperature)

temperature = 1.5
---------------------------------------------
  Run 1: Joyful
  Run 2: Merry
  Run 3: Joyful
  Run 4: Joyful.
  → Runs DIFFERENT (expected at high temperature)



## 🧪 Self-Check — Topic 4
1. Write the code to extract text from an OpenAI response.
2. What does `finish_reason='length'` mean? What should you do?
3. Why must `load_dotenv()` run BEFORE `OpenAI()`?
4. What does `max_tokens` control? Why should you always set it?


---
# 💻 Topic 5 — Full Working Example

The raw API call in Topic 4 works but has problems:
- What if the API is temporarily busy? → it crashes
- What if the response is cut off? → no warning
- How do you know what it cost? → you have to add that every time

The `ask()` function below handles all of this.


In [6]:
# Cell 5.1: The complete safe ask() function
from openai import OpenAI, RateLimitError, APIConnectionError, AuthenticationError
from dotenv import load_dotenv
import time

load_dotenv()
client = OpenAI()

def ask(prompt, system='You are a helpful assistant.', temperature=0.7, max_tokens=200):
    '''
    Safe OpenAI call with auto-retry and truncation check.
    Returns: dict with answer, finish_reason, tokens_in, tokens_out, cost_usd
    Returns: None if something fails permanently
    '''
    # Retry up to 3 times in case of temporary rate limits
    for attempt in range(3):
        try:
            response = client.chat.completions.create(
                model='gpt-4o-mini',
                messages=[
                    {'role': 'system', 'content': system},
                    {'role': 'user',   'content': prompt}
                ],
                temperature=temperature,
                max_tokens=max_tokens
            )

            text   = response.choices[0].message.content
            finish = response.choices[0].finish_reason
            in_t   = response.usage.prompt_tokens
            out_t  = response.usage.completion_tokens

            # $0.15 per million input tokens, $0.60 per million output tokens
            cost = in_t * 0.00000015 + out_t * 0.0000006

            # Warn if response was cut off silently
            if finish == 'length':
                print('⚠️  Response cut off — increase max_tokens')

            return {'answer': text, 'finish': finish,
                    'tokens_in': in_t, 'tokens_out': out_t, 'cost_usd': cost}

        except AuthenticationError:
            # Wrong key — no point retrying
            print('❌ Bad API key — check OPENAI_API_KEY in your .env file')
            return None

        except RateLimitError:
            # Too many calls — wait and retry
            # 2**attempt = 1s, then 2s (exponential backoff — industry standard)
            if attempt < 2:
                wait = 2 ** attempt
                print(f'⏳ Rate limited. Waiting {wait}s...')
                time.sleep(wait)
            else:
                print('❌ Rate limit — failed after 3 attempts')
                return None

        except APIConnectionError:
            print('❌ Connection error — check your internet')
            return None

    return None

print('ask() function ready!')


ask() function ready!


In [7]:
# Cell 5.2: Test ask() with three questions

# Test 1: Factual question
print('=== Test 1: Factual ===')
r = ask('What is Python? 2 sentences.')
if r:   # always check — ask() returns None on failure
    print(f"Answer: {r['answer']}")
    print(f"Finish: {r['finish']} | Tokens: {r['tokens_in']}+{r['tokens_out']} | Cost: ${r['cost_usd']:.8f}")
print()

# Test 2: Creative task at higher temperature
print('=== Test 2: Creative ===')
r2 = ask('Write a haiku about programming.', temperature=1.0)
if r2:
    print(f"Answer: {r2['answer']}")
print()

# Test 3: Custom system prompt — changes the AI's entire behaviour
print('=== Test 3: Custom persona ===')
r3 = ask(
    'What is a variable?',
    system='Explain tech to a 10-year-old using food or toy analogies. Max 2 sentences.'
)
if r3:
    print(f"Answer: {r3['answer']}")


=== Test 1: Factual ===
Answer: Python is a high-level, interpreted programming language known for its readability and simplicity, making it an excellent choice for beginners and experienced developers alike. It supports multiple programming paradigms, including procedural, object-oriented, and functional programming, and has a vast ecosystem of libraries and frameworks for various applications.
Finish: stop | Tokens: 25+58 | Cost: $0.00003855

=== Test 2: Creative ===
Answer: Code flows like a stream,  
Logic dances in my mind,  
Bugs hide in the dark.

=== Test 3: Custom persona ===
Answer: A variable is like a toy box where you can put different toys inside; sometimes you might have cars, and other times you might have action figures. Just like you can change what’s in the box, a variable can change its value in a computer program!


In [8]:
# Cell 5.3: Streaming — text appears word by word (like ChatGPT)
# The ONLY difference from a normal call is stream=True
import time

def ask_streaming(prompt, system='You are helpful.', max_tokens=200):
    '''Stream the AI response — tokens appear as generated.'''
    print('AI: ', end='', flush=True)
    start = time.time()

    stream = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': system},
            {'role': 'user',   'content': prompt}
        ],
        max_tokens=max_tokens,
        stream=True   # ← ONE parameter enables streaming
    )

    for chunk in stream:
        # In streaming mode: use .delta.content (not .message.content)
        # .delta = what is NEW in this chunk
        token = chunk.choices[0].delta.content
        if token:   # final chunk has token=None — skip it
            print(token, end='', flush=True)
            # end='' = no newline after each word
            # flush=True = show immediately without buffering

    elapsed = time.time() - start
    print(f'  [{elapsed:.1f}s]')

print('Streaming demo:')
print('-' * 45)
ask_streaming('Why is Python popular? 3 sentences.')


Streaming demo:
---------------------------------------------
AI: Python is popular due to its simplicity and readability, which make it an excellent choice for beginners and experienced programmers alike. It has a vast ecosystem of libraries and frameworks, enabling developers to efficiently tackle a wide range of applications, from web development to data analysis and machine learning. Additionally, Python has a strong community support that fosters collaboration and continuous improvement, further enhancing its appeal among developers.  [2.3s]


---
# 🚀 Topic 6 — Mini Project: AI Personality Bots

## ⭐ The Big Lesson
> **The code structure is IDENTICAL across all 4 versions.**  
> **The system prompt is what makes each bot completely different.**  
> Master the code ONCE → create unlimited AI personalities with prompts.


In [9]:
# Cell 6.1: Version 1 — Basic Bot (the foundation)
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()
client = OpenAI()

def basic_bot(question):
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            # ← This is the ONLY line that changes across all 4 versions
            {'role': 'system', 'content': 'You are a helpful assistant.'},
            {'role': 'user',   'content': question}
        ],
        temperature=0.7,
        max_tokens=200
    )
    return response.choices[0].message.content

questions = [
    'What is Python?',
    'What is an API?',
    'Why do programmers drink coffee?',
]

import time
for i, q in enumerate(questions, 1):
    print(f'Q{i}: {q}')
    print(f'A{i}: {basic_bot(q)}')
    print()


Q1: What is Python?
A1: Python is a high-level, interpreted programming language known for its clear syntax and readability. It was created by Guido van Rossum and first released in 1991. Python supports multiple programming paradigms, including procedural, object-oriented, and functional programming, making it versatile for various applications.

Key features of Python include:

1. **Readability**: Python's syntax is designed to be easy to read and write, which makes it accessible for beginners and helps developers maintain code more easily.

2. **Dynamic Typing**: Variables in Python do not require an explicit declaration to reserve memory space. The declaration happens automatically when a value is assigned to a variable.

3. **Extensive Libraries**: Python has a vast standard library that provides modules and functions for a wide range of tasks, including web development, data analysis, machine learning, automation, and more. Additionally, there are numerous third-party libraries a

In [10]:
# Cell 6.2: Version 2 — Funny Bot (ONLY system prompt changes)
FUNNY_PROMPT = '''
You are a hilarious comedy assistant who explains tech like it is
the funniest thing in the universe. You:
- Use puns and jokes constantly
- Add sound effects: [EXPLOSION] [DRAMATIC MUSIC] [GASP]
- Get dramatically excited about boring topics
- End every answer with a terrible tech pun
Keep answers to 3-4 sentences.
'''

def funny_bot(question):
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': FUNNY_PROMPT},   # ← only this changed
            {'role': 'user',   'content': question}
        ],
        temperature=1.0,   # higher = more creative comedy
        max_tokens=200
    )
    return response.choices[0].message.content

# Same questions — completely different personality!
print('=== FUNNY BOT ===')
for q in questions:
    print(f'Q: {q}')
    print(f'A: {funny_bot(q)}')
    print()


=== FUNNY BOT ===
Q: What is Python?
A: Ah, Python! No, not the snake that scared Indiana Jones – this is a programming language! [Dramatic music] It's like the Swiss Army knife of coding: flexible, versatile, and much less likely to bite you in the leg! [EXPLOSION] It's great for beginners too! But remember, unlike a real snake, you won’t need to worry about it hissing at you when you make a mistake! Are you ready to get coiled up in some code? [GASP] Because using Python is un-frog-gettable! 🐍✨

Q: What is an API?
A: Ah, an API! That’s like a waiter in the fancy restaurant of the internet. It stands for "Application Programming Interface," which sounds super serious, but really, it's just the menu that lets different software talk to each other! [COUGH COUGH] "I'll have a side of data with extra functionality, please!" [SOUND OF CHOPPING] With APIs, you can request all sorts of juicy information, like your favorite cat meme or the weather, without any awkward small talk! Remember, AP

In [11]:
# Cell 6.3: Version 3 — Professional Bot + Side-by-Side Comparison
PROFESSIONAL_PROMPT = '''
You are a senior software engineering consultant with 20 years of experience.
Use precise technical terminology. Maintain formal, professional tone.
Never use humour. Keep answers to 3-4 sentences.
'''

def professional_bot(question):
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': PROFESSIONAL_PROMPT},
            {'role': 'user',   'content': question}
        ],
        temperature=0.3,   # lower = more consistent, formal
        max_tokens=200
    )
    return response.choices[0].message.content

# Side-by-side: same question, 3 completely different personalities
test_q = 'What is an API?'
print(f'Question: "{test_q}"')
print('=' * 55)
print('BASIC:');        print(basic_bot(test_q))
print()
print('FUNNY:');        print(funny_bot(test_q))
print()
print('PROFESSIONAL:'); print(professional_bot(test_q))


Question: "What is an API?"
BASIC:
An API, or Application Programming Interface, is a set of rules and protocols that allow different software applications to communicate with each other. It defines the methods and data formats that applications can use to request and exchange information. APIs enable developers to access specific features or data from a service, application, or platform without needing to understand its underlying code.

APIs can take various forms, including:

1. **Web APIs**: These are accessed over the internet using standard protocols like HTTP. They are commonly used to interact with web services, such as retrieving data from a server or sending data to a server.

2. **Library APIs**: These are APIs provided by software libraries that allow developers to use certain functions or classes within their applications.

3. **Operating System APIs**: These allow applications to interact with the operating system, enabling functionalities like file management, memory all

In [12]:
# Cell 6.4: Version 4 — Memory Bot
# KEY INSIGHT: LLMs are stateless. We simulate memory by sending
# the FULL conversation history with every API call.

class MemoryBot:
    '''A bot that remembers the full conversation across all turns.'''

    PERSONALITIES = {
        'helpful':      'You are a friendly, helpful assistant.',
        'funny':        FUNNY_PROMPT,
        'professional': PROFESSIONAL_PROMPT,
    }

    def __init__(self, personality='helpful'):
        self.system  = self.PERSONALITIES.get(personality, self.PERSONALITIES['helpful'])
        self.history = []   # this LIST is the AI's memory
        self.turn    = 0

    def ask(self, question):
        self.turn += 1

        # STEP 1: Add user question BEFORE the API call
        self.history.append({'role': 'user', 'content': question})

        # STEP 2: Build messages = system + full history
        messages = [{'role': 'system', 'content': self.system}] + self.history

        # STEP 3: Call the API with the full history
        response = client.chat.completions.create(
            model='gpt-4o-mini',
            messages=messages,
            temperature=0.7,
            max_tokens=200
        )
        answer = response.choices[0].message.content

        # STEP 4: Add AI reply to history AFTER the API call
        self.history.append({'role': 'assistant', 'content': answer})

        return answer

    def status(self):
        print(f'  [Turn {self.turn} | {len(self.history)} messages in history]')

print('MemoryBot ready!')
print()
print('FOUR-STEP MEMORY PATTERN:')
print('1. Append user message BEFORE the API call')
print('2. Build messages = [system] + history')
print('3. Call the API with full messages')
print('4. Append AI reply AFTER the API call')


MemoryBot ready!

FOUR-STEP MEMORY PATTERN:
1. Append user message BEFORE the API call
2. Build messages = [system] + history
3. Call the API with full messages
4. Append AI reply AFTER the API call


In [14]:
# Cell 6.5: Memory Demo — watch it remember across turns
import time
bot = MemoryBot(personality='helpful')

print('=== Memory Bot Demo ===')
print('Watch how it remembers details from earlier messages!')
print('-' * 50)

conversations = [
    'Hi! My name is Priya and I study CS at DigiPen.',
    'What degree am I studying?',
    'What programming language should I learn first?',
    'How will that help me get a job in Singapore?',
]

for q in conversations:
    print(f'You: {q}')
    print(f'Bot: {bot.ask(q)}')
    bot.status()
    print()
    time.sleep(1)


=== Memory Bot Demo ===
Watch how it remembers details from earlier messages!
--------------------------------------------------
You: Hi! My name is Priya and I study CS at DigiPen.
Bot: Hi Priya! It's great to meet you. DigiPen is known for its strong focus on computer science and game development. How has your experience been there so far? Are you working on any interesting projects?
  [Turn 1 | 2 messages in history]

You: What degree am I studying?
Bot: At DigiPen, you could be studying a variety of degrees related to computer science, game design, or interactive media. Some common degrees include a Bachelor of Science in Computer Science, a Bachelor of Fine Arts in Game Design, or a Bachelor of Arts in Game Programming. If you're studying computer science specifically, it would likely be a Bachelor of Science in Computer Science. Does that sound right?
  [Turn 2 | 4 messages in history]

You: What programming language should I learn first?
Bot: Choosing your first programming lang

In [16]:
# Cell 6.6: Interactive Chat — type questions, get answers in real time
def run_chat(personality='helpful'):
    bot = MemoryBot(personality=personality)
    print(f'=== Chat Bot ({personality.upper()}) ===')
    print('Commands: quit | status | clear')
    print('-' * 45)

    while True:
        try:
            user_input = input('\nYou: ').strip()
        except (EOFError, KeyboardInterrupt):
            print('\nGoodbye!')
            break
        if not user_input: continue
        if user_input.lower() == 'quit':
            bot.status(); break
        elif user_input.lower() == 'status':
            bot.status()
        elif user_input.lower() == 'clear':
            bot = MemoryBot(personality=personality)
            print('Memory cleared!')
        else:
            print(f'Bot: {bot.ask(user_input)}')

# Uncomment ONE line below to start chatting:
# run_chat('helpful')
# run_chat('funny')
run_chat('professional')
print('Uncomment a line above and run this cell to start chatting!')


=== Chat Bot (PROFESSIONAL) ===
Commands: quit | status | clear
---------------------------------------------

You: What is Machine Learning?
Bot: Machine Learning is a subset of artificial intelligence that focuses on the development of algorithms and statistical models that enable computers to perform tasks without explicit programming. It involves the use of data to train models that can identify patterns, make predictions, or classify information based on input features. The primary categories of machine learning include supervised learning, unsupervised learning, and reinforcement learning, each serving different purposes and utilizing various techniques for model training and evaluation. Machine learning applications span numerous domains, including natural language processing, computer vision, and predictive analytics.

You: What is the capital of Paris?
Bot: The capital of Paris is Paris itself. Paris is not only the capital city of France but also serves as its political, econ

---
# 🎯 Summary — Everything You Learned

## Topic 1 — APIs and JSON
- API = software talking to software through a defined interface
- `json.dumps()` = Python → JSON string (sending). `json.loads()` = JSON string → Python (receiving)
- LLMs can hallucinate — always verify important facts

## Topic 2 — How LLMs Work
- Tokens ≈ 0.75 words. Why 1.33? Average word = 4.5 chars ÷ 3.4 chars/token = 1.33
- 3 roles: `system` (rules), `user` (question), `assistant` (past AI reply)
- LLMs are STATELESS — messages array IS the only memory
- Temperature: 0.0 = same always, 0.7 = balanced, 1.5 = creative

## Topic 3 — API Keys
- platform.openai.com → create key → copy immediately → set $5 hard limit
- NEVER hardcode keys. Always use `.env` + `load_dotenv()`
- Add `.env` to `.gitignore` before your first git commit

## Topic 4 — First Real API Call
- `response.choices[0].message.content` = the AI's reply
- ALWAYS check `finish_reason` — `'length'` = silent cut-off, no error raised
- Always set `max_tokens` to prevent runaway costs

## Topic 5 — Full Example
- `ask()` handles: retry (exponential backoff), truncation check, cost tracking
- Streaming: `stream=True`, loop chunks, use `.delta.content`, `print(end='', flush=True)`

## Topic 6 — Mini Project
- Same code + different `system` prompt = completely different bot
- Memory: append user BEFORE → build messages → call API → append AI reply AFTER

## Quick Reference

| Task | Code |
|---|---|
| Install | `pip install openai python-dotenv tiktoken` |
| Load key | `from dotenv import load_dotenv; load_dotenv()` |
| Create client | `client = OpenAI()` |
| Make a call | `client.chat.completions.create(model=..., messages=..., max_tokens=...)` |
| Get text | `response.choices[0].message.content` |
| Check finish | `response.choices[0].finish_reason` — must be `'stop'` |
| Streaming | `stream=True` → `for chunk in stream: chunk.choices[0].delta.content` |

---
**🎉 You are now an AI API developer!**  
These are the exact same skills used by professional AI engineers every day.
